# DEE v6 — SIM 23: Sweep adiabático sobre S³ vs R³**Autor:** Juan Pablo Bruschi (2026)**Versión:** v6 SIM 23 (complemento de SIM 22)**Estado:** test cosmológico anharmónico en topología esférica positiva---## Pregunta científicaSIM 22 testeó el sector topológico anharmónico para T³ vs R³ y encontró:- Control armónico g₄=0: α = −1,000 exacto (escala radiación-like)- Régimen anharmónico g₄=5000: α = −1,291 → w_top = +0,430- **Conclusión:** el sector topológico T³ vive en w ≈ +1/3, no cerca de w_DE = −0,70La pregunta abierta: ¿este resultado es universal sobre cualquier topología cerrada, o depende de la topología específica?S³ es el contraste más natural por dos razones:1. **Sector infrarrojo cualitativamente distinto**: cuadruplete (n=1, multiplicidad 4) en S³ vs sexteto en T³, λ_1 = 3 en convención n(n+2) vs (2π)² en T³.2. **Curvatura positiva intrínseca**: S³ tiene Ric > 0 (Cap. 4 §4.3), a diferencia de T³ que tiene Ric = 0. La frustración geométrica F_S³ / F_T³ = 1,64× medida en SIM 18 (Cap. 4 §4.5) confirma que las dos topologías son cualitativamente distintas en el modelo.Si la corrección Hartree 3·g₄·⟨φ²⟩ es sensible a la estructura del sector IR, S³ y T³ podrían dar resultados cuantitativamente diferentes para α(g₄).## Setup del test- N = 1200 puntos sobre S³ ⊂ R⁴ con muestreo cuasi-regular (mismo método que SIM 10/15)- N ≈ 1200 puntos sobre R³ con BC abiertas (control)- Radio R = {0,70, 0,85, 1,00, 1,20, 1,40} — escalado adiabático de S³ (las geodésicas se vuelven R · arccos)- g₄ = {0, 100, 1000, 5000} — mismo rango Hartree-válido que SIM 22- Hartree autoconsistente igual que SIM 21/22- Observable: ΔE(R, g₄) = E_zp_S³(R, g₄) − E_zp_R³(R, g₄)- Extracción α_S³(g₄) por ajuste log-log de |ΔE| vs R## Predicciones de referencia| Resultado | Interpretación ||---|---|| α_S³(g₄_max) ≈ α_T³(g₄_max) ≈ −1,29 | Sector topológico universal — robustez del resultado SIM 22 || α_S³(g₄_max) ≠ α_T³(g₄_max) | Dependencia topológica del régimen anharmónico — novedad cuantitativa || α_S³(g₄_max) → +2,1 | Mecanismo específico de S³ podría conectar con w = −0,70 || α_S³(g₄=0) ≈ −1 (control) | Validación del teorema de escala armónico sobre S³ |## Tiempo estimado5 R × 4 g₄ × 2 topologías × Hartree iterativo. N=1200 idéntico a SIM 22. Estimación: **25–35 min** en Colab CPU.

---## 1. Setup

In [ ]:
import os, time, json
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

OUT_DIR = '/content/dee_v6_S3_sweep'
os.makedirs(OUT_DIR, exist_ok=True)

def out_path(name): return os.path.join(OUT_DIR, name)

print(f'Outputs:  {OUT_DIR}')
print(f'NumPy:    {np.__version__}')

In [ ]:
# Paleta
BG  = '#0d1117'
CW  = '#ecf0f1'
CY  = '#f1c40f'
CG  = '#27ae60'
CB  = '#2980b9'
CR  = '#e74c3c'
CO  = '#e67e22'
CGR = '#7f8c8d'
CP  = '#9b59b6'

def setup_dark_axes(ax):
    ax.set_facecolor(BG)
    ax.tick_params(colors=CGR)
    for s in ax.spines.values():
        s.set_color('#2c3e50')

def new_figure(nrows, ncols, figsize):
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    fig.patch.set_facecolor('#0a0a1a')
    if hasattr(axes, 'flat'):
        for ax in axes.flat: setup_dark_axes(ax)
    else:
        setup_dark_axes(axes)
    return fig, axes

---## 2. Configuración global

In [ ]:
# ───── parámetros ─────
N_TARGET     = 1200          # puntos S³ y R³
R_VALS       = [0.70, 0.85, 1.00, 1.20, 1.40]  # mismo rango que SIM 22
G4_VALS      = [0.0, 100.0, 1000.0, 5000.0]    # mismo rango Hartree
JITTER_S3    = 0.10          # jitter óptimo de SIM 10 (Cap. 4 §4.2)
SEED_BASE    = 42
K_NEIGHBORS  = 30
EPS_REG      = 1e-6
HARTREE_MAX  = 25
HARTREE_TOL  = 1e-5

print(f'N        = {N_TARGET}')
print(f'R_VALS   = {R_VALS}')
print(f'G4_VALS  = {G4_VALS}')
print(f'Total puntos: {len(R_VALS) * len(G4_VALS) * 2} (R × g₄ × topologías)')

---## 3. Funciones coreLas funciones de muestreo cuasi-regular de S³, distancias geodésicas, y Laplaciano sparse son las del notebook `Tests_T3/DEE_T3_S3_final.ipynb` del repositorio — mismo método que produjo SIM 10 y SIM 15 del documento DEE v6.**Punto clave para el sweep adiabático sobre S³:** la red base se muestrea **una sola vez** en S³ unitario (radio 1). Para "expandir" el espacio (análogo a L → L·(1+ε) en T³), escalo el radio R. Las distancias geodésicas son R · arccos(x_i · x_j), las cuales bajo R → R·(1+ε) escalan proporcionalmente, igual que las distancias periódicas de T³ bajo L → L·(1+ε). Es exactamente el mismo experimento físico, sólo cambiado el espacio.

In [ ]:
from scipy.sparse import csr_matrix, diags

def sample_S3_quasi_regular(N_target, jitter_fraction=0.1, seed=42):
    """Muestra cuasi-regular de S³ ⊂ R⁴ (del repo Tests_T3)."""
    rng = np.random.default_rng(seed)
    x = rng.normal(size=(N_target, 4))
    x = x / np.linalg.norm(x, axis=1, keepdims=True)
    # Repulsión Lloyd simplificada
    for _ in range(3):
        D = np.arccos(np.clip(x @ x.T, -1, 1))
        np.fill_diagonal(D, np.inf)
        nearest = np.argpartition(D, 10, axis=1)[:, :10]
        displacements = np.zeros_like(x)
        for i in range(N_target):
            for j in nearest[i]:
                direction = x[i] - x[j]
                direction -= np.dot(direction, x[i]) * x[i]
                d = D[i, j]
                if d > 0:
                    displacements[i] += direction / (d**2 * 100)
        x = x + displacements
        x = x / np.linalg.norm(x, axis=1, keepdims=True)
    if jitter_fraction > 0:
        typical_spacing = np.pi * N_target**(-1/3)
        jitter_amount = jitter_fraction * typical_spacing
        tangent_dirs = rng.normal(size=(N_target, 4))
        tangent_dirs -= np.sum(tangent_dirs * x, axis=1, keepdims=True) * x
        tangent_dirs = tangent_dirs / np.linalg.norm(tangent_dirs, axis=1, keepdims=True)
        x = x + jitter_amount * tangent_dirs
        x = x / np.linalg.norm(x, axis=1, keepdims=True)
    return x

def geodesic_distance_S3(points_unit, R=1.0):
    """Distancia geodésica en S³ de radio R: d = R · arccos(x·y)."""
    dots = points_unit @ points_unit.T
    dots = np.clip(dots, -1.0, 1.0)
    return R * np.arccos(dots)

def make_R3_control(N_target, R, seed=42):
    """Muestreo random uniforme en caja [0, R]³ — control R³ (BC abiertas)."""
    rng = np.random.default_rng(seed)
    return rng.uniform(0, R, size=(N_target, 3))

def open_distance_matrix(coords):
    D = cdist(coords, coords)
    np.fill_diagonal(D, np.inf)
    return D

def build_laplacian_dense(D, k_neighbors=K_NEIGHBORS):
    """Laplaciano DENSO con k-NN y kernel 1/d². Para N=1200 cabe en memoria."""
    N = D.shape[0]
    W = np.zeros((N, N))
    for i in range(N):
        idx = np.argsort(D[i])[:k_neighbors]
        for j in idx:
            if D[i, j] > 0 and np.isfinite(D[i, j]):
                W[i, j] = 1.0 / D[i, j]**2
    W = np.maximum(W, W.T)
    return np.diag(W.sum(axis=1)) - W

def hartree_self_consistent(K_base, g4, max_iter=HARTREE_MAX, tol=HARTREE_TOL,
                             eps_reg=EPS_REG):
    """Hartree autoconsistente — misma implementación que SIM 21/22."""
    N = K_base.shape[0]
    K_reg = K_base + eps_reg * np.eye(N)
    K_eff = K_reg.copy()
    phi_sq_prev = None
    chi_h = 0.0
    for it in range(max_iter):
        lam, vec = np.linalg.eigh(K_eff)
        lam = np.maximum(lam, eps_reg)
        inv_sqrt_diag = (vec**2 @ (1.0 / np.sqrt(lam)))
        phi_sq = 0.5 * inv_sqrt_diag
        correction_norm = np.linalg.norm(3.0 * g4 * phi_sq)
        K_base_norm = np.linalg.norm(K_base)
        chi_h = correction_norm / max(K_base_norm, 1e-10)
        K_eff_new = K_reg + 3.0 * g4 * np.diag(phi_sq)
        if phi_sq_prev is not None:
            change = np.max(np.abs(phi_sq - phi_sq_prev) / (np.abs(phi_sq_prev) + 1e-10))
            if change < tol:
                break
        K_eff = K_eff_new
        phi_sq_prev = phi_sq.copy()
    lam_final = np.linalg.eigvalsh(K_eff)
    lam_final = np.maximum(lam_final, eps_reg)
    return lam_final, chi_h, it + 1

def E_zp_from_lambda(lam):
    return 0.5 * np.sum(np.sqrt(np.maximum(lam, 0.0)))

---## 4. Muestreos base (una sola vez)Se generan **una vez** la red S³ unitaria (que después se escala por R) y la red R³ unitaria (que también se escala por R). Esto garantiza que solo cambia la escala absoluta, no la geometría relativa — exactamente análogo al sweep adiabático de SIM 22.

In [ ]:
print('Generando muestreo cuasi-regular de S³...')
points_S3 = sample_S3_quasi_regular(N_TARGET, jitter_fraction=JITTER_S3, seed=SEED_BASE)
print(f'  shape S³: {points_S3.shape}  (norma media: {np.linalg.norm(points_S3, axis=1).mean():.5f})')

print('Generando muestreo R³ control...')
points_R3_unit = make_R3_control(N_TARGET, R=1.0, seed=SEED_BASE)
print(f'  shape R³: {points_R3_unit.shape}')

# Diagnóstico del muestreo S³: el primer autovalor armónico debería ser cercano a 3
# (convención n(n+2), λ_1 = 3 sobre S³ de radio 1)
D_test = geodesic_distance_S3(points_S3, R=1.0)
L_test = build_laplacian_dense(D_test)
lam_test = np.linalg.eigvalsh(L_test)
print(f'\nDiagnóstico S³ unitario:')
print(f'  λ_0 (modo cero) = {lam_test[0]:.6f}')
print(f'  λ_1 a λ_4 (cuadruplete esperado) = {lam_test[1:5]}')
print(f'  Dispersión cuadruplete: {(lam_test[4]-lam_test[1])/lam_test[4]*100:.2f}%')
print(f'  λ_5/λ_1 = {lam_test[5]/lam_test[1]:.3f}  (teórico 8/3 ≈ 2,667)')

---## 5. Función principal: ΔE(R, g₄) por sweep adiabático

In [ ]:
def medir_punto_R(R, g4, verbose=False):
    """Mide E_zp en S³ (radio R) y R³ (caja [0,R]³) con anharmonicidad g₄."""
    # S³ a radio R
    D_S3 = geodesic_distance_S3(points_S3, R=R)
    L_S3 = build_laplacian_dense(D_S3)
    lam_S3, chi_S3, it_S3 = hartree_self_consistent(L_S3, g4)
    E_S3 = E_zp_from_lambda(lam_S3)

    # R³ caja [0,R]³ (control)
    coords_R3 = points_R3_unit * R
    D_R3 = open_distance_matrix(coords_R3)
    L_R3 = build_laplacian_dense(D_R3)
    lam_R3, chi_R3, it_R3 = hartree_self_consistent(L_R3, g4)
    E_R3 = E_zp_from_lambda(lam_R3)

    dE = E_S3 - E_R3

    if verbose:
        print(f'  R={R:.2f}, g₄={g4:>5.0f}:  E_S³={E_S3:.2f}  E_R³={E_R3:.2f}  ΔE={dE:.2f}')
        print(f'    χ_H(S³)={chi_S3:.4f}  χ_H(R³)={chi_R3:.4f}  iter S³={it_S3}  R³={it_R3}')

    return dict(R=R, g4=g4, V=R**3,
                E_S3=E_S3, E_R3=E_R3, dE=dE,
                chi_S3=chi_S3, chi_R3=chi_R3,
                iter_S3=it_S3, iter_R3=it_R3)

---## 6. Barrido principal con persistencia

In [ ]:
SIM23_FILE = out_path('sim23_progress.json')

if os.path.exists(SIM23_FILE):
    with open(SIM23_FILE) as f:
        results = json.load(f)
    print(f'Progreso previo: {len(results)} puntos cargados')
else:
    results = []

ya_hechos = set((r['R'], r['g4']) for r in results)
faltantes = [(R, g4) for R in R_VALS for g4 in G4_VALS
             if (R, g4) not in ya_hechos]
total = len(R_VALS) * len(G4_VALS)
print(f'Puntos totales: {total}, faltantes: {len(faltantes)}')

t_start = time.time()
for k, (R, g4) in enumerate(faltantes):
    print(f'\n[{k+1}/{len(faltantes)}] R={R:.2f}, g₄={g4:.0f} ...')
    t0 = time.time()
    pt = medir_punto_R(R, g4, verbose=True)
    dt = time.time() - t0
    print(f'    → {dt:.1f}s')
    results.append(pt)
    with open(SIM23_FILE, 'w') as f:
        json.dump(results, f, indent=1)

print(f'\nTotal: {len(results)} puntos en {time.time()-t_start:.1f}s')

---## 7. Análisis: α_S³(g₄) y comparación con α_T³ de SIM 22

In [ ]:
import numpy as np

# Organizar por g₄
by_g4 = {}
for r in results:
    by_g4.setdefault(r['g4'], []).append(r)
for g4 in by_g4:
    by_g4[g4] = sorted(by_g4[g4], key=lambda r: r['R'])

# Ajuste log-log
print(f'{"g₄":>8s}  {"α_S³":>10s}  {"R²":>8s}  {"w_top=-α/3":>13s}  {"χ_H max":>10s}')
print('─' * 65)

ajustes = {}
for g4 in sorted(by_g4.keys()):
    pts = by_g4[g4]
    Rs   = np.array([p['R']  for p in pts])
    dEs  = np.array([abs(p['dE']) for p in pts])
    chis = max(max(p['chi_S3'] for p in pts), max(p['chi_R3'] for p in pts))
    valid = dEs > 0
    if valid.sum() < 3:
        alpha = np.nan; R2 = np.nan
    else:
        lR = np.log(Rs[valid]); ldE = np.log(dEs[valid])
        coef = np.polyfit(lR, ldE, 1)
        alpha = coef[0]
        ldE_pred = np.polyval(coef, lR)
        ss_res = np.sum((ldE - ldE_pred)**2)
        ss_tot = np.sum((ldE - ldE.mean())**2)
        R2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    w_top = -alpha / 3 if not np.isnan(alpha) else np.nan
    ajustes[g4] = dict(alpha=alpha, R2=R2, w_top=w_top, chi_max=chis,
                        Rs=Rs, dEs=dEs,
                        signed_dE=np.array([p['dE'] for p in pts]))
    print(f'{g4:>8.0f}  {alpha:>+10.4f}  {R2:>8.4f}  {w_top:>+13.4f}  {chis:>10.4f}')

In [ ]:
# Análisis específico
print('\n═══ CONTROL ARMÓNICO (g₄ = 0) ═══')
alpha_arm = ajustes[0.0]['alpha']
w_arm     = ajustes[0.0]['w_top']
print(f'  α_S³(g₄=0) medido            = {alpha_arm:+.4f}')
print(f'  α esperado (escala armónica) = −1,00')
print(f'  Diferencia |α − (−1)|        = {abs(alpha_arm + 1):.4f}')
print(f'  w_top(g₄=0)                  = {w_arm:+.4f}  (esperado +1/3)')
print(f'  → {"✓ Reproduce escala armónica" if abs(alpha_arm + 1) < 0.25 else "⚠ NO reproduce"}')

print('\n═══ RÉGIMEN ANHARMÓNICO ═══')
g4_max_valid = max(g for g in ajustes.keys() if ajustes[g]['chi_max'] < 0.5 and g > 0)
alpha_max = ajustes[g4_max_valid]['alpha']
w_max     = ajustes[g4_max_valid]['w_top']
delta_alpha = alpha_max - alpha_arm
delta_w     = w_max - w_arm
print(f'  α_S³(g₄ = {g4_max_valid:.0f}) = {alpha_max:+.4f}  (χ_H = {ajustes[g4_max_valid]["chi_max"]:.3f})')
print(f'  w_top(g₄ = {g4_max_valid:.0f})  = {w_max:+.4f}')
print(f'  Δα   = {delta_alpha:+.4f}')
print(f'  Δw   = {delta_w:+.4f}')

# Comparación con SIM 22 (valores fijos del archivo previo)
ALPHA_T3_REF = {0.0: -1.0000, 100.0: -1.0105, 1000.0: -1.0869, 5000.0: -1.2910}
W_T3_REF = {g: -a/3 for g, a in ALPHA_T3_REF.items()}

print('\n═══ COMPARACIÓN S³ (SIM 23) vs T³ (SIM 22) ═══')
print(f'{"g₄":>8s}  {"α_S³":>10s}  {"α_T³ ref":>10s}  {"Δα":>10s}  '
      f'{"w_S³":>10s}  {"w_T³":>10s}  {"Δw":>10s}')
print('─' * 78)
diffs_alpha = []
for g4 in sorted(by_g4.keys()):
    if g4 not in ALPHA_T3_REF: continue
    aS = ajustes[g4]['alpha']; aT = ALPHA_T3_REF[g4]
    wS = -aS/3; wT = W_T3_REF[g4]
    print(f'{g4:>8.0f}  {aS:>+10.4f}  {aT:>+10.4f}  {aS-aT:>+10.4f}  '
          f'{wS:>+10.4f}  {wT:>+10.4f}  {wS-wT:>+10.4f}')
    diffs_alpha.append((g4, aS, aT, aS-aT))

# Veredicto
print('\n═══ VEREDICTO TOPOLOGÍA-DEPENDENCIA ═══')
delta_t3_s3_max = max(abs(d[3]) for d in diffs_alpha)
delta_t3_s3_anharm = max(abs(d[3]) for d in diffs_alpha if d[0] > 0)
print(f'|α_S³ − α_T³| máximo a g₄ > 0     = {delta_t3_s3_anharm:.4f}')
print(f'|α_S³ − α_T³| en control (g₄=0)    = {abs(diffs_alpha[0][3]) if diffs_alpha[0][0]==0 else "n/a":}')
if delta_t3_s3_anharm < 0.10:
    print(f'  → RESULTADO TOPOLÓGICAMENTE UNIVERSAL')
    print(f'  El sector topológico anharmónico produce w ≈ +1/3 sobre ambas topologías cerradas.')
    print(f'  Refuerza el resultado SIM 22: la predicción w_DE = −0,70 del Cap. 6 §6.4 (homogéneo)')
    print(f'  no se ve afectada por el sector topológico, ni en T³ ni en S³.')
elif delta_t3_s3_anharm < 0.30:
    print(f'  → DEPENDENCIA TOPOLÓGICA MODERADA')
    print(f'  Diferencia detectable entre T³ y S³ pero pequeña.')
    print(f'  Sugiere efecto sub-líder que podría refinarse con más estadística.')
else:
    print(f'  → DEPENDENCIA TOPOLÓGICA FUERTE')
    print(f'  El régimen anharmónico distingue cuantitativamente entre T³ y S³.')
    print(f'  Resultado nuevo: el sustrato responde de manera específica a la topología bajo anharmonicidad.')

---## 8. Visualización

In [ ]:
fig, axes = new_figure(2, 2, figsize=(14, 10))
colors = plt.cm.viridis(np.linspace(0, 0.85, len(G4_VALS)))

# Panel A: |ΔE| vs R log-log para S³
ax1 = axes[0, 0]
for g4, col in zip(sorted(by_g4.keys()), colors):
    aj = ajustes[g4]
    ax1.loglog(aj['Rs'], aj['dEs'], 'o-', color=col, ms=9, lw=2,
               label=f'g₄={g4:.0f}  α={aj["alpha"]:+.3f}')
ax1.set_xlabel('R (radio S³)', color=CW)
ax1.set_ylabel('|ΔE| = |E_S³ − E_R³|', color=CW)
ax1.set_title('Sweep adiabático sobre S³: ΔE vs R\n(N fijo, V = R³ análogo a SIM 22)', color=CW, fontweight='bold')
ax1.legend(facecolor=BG, labelcolor=CW, fontsize=9)
ax1.grid(True, alpha=0.15, which='both')

# Panel B: comparación α_S³ vs α_T³
ax2 = axes[0, 1]
g4s_plot = np.array(sorted(by_g4.keys()))
alphas_S3 = np.array([ajustes[g]['alpha'] for g in g4s_plot])
alphas_T3_ref = np.array([ALPHA_T3_REF.get(g, np.nan) for g in g4s_plot])

ax2.semilogx(np.maximum(g4s_plot, 0.1), alphas_S3, 'o-', color=CR, ms=12, lw=2.5,
             label='α_S³ (SIM 23, este test)')
ax2.semilogx(np.maximum(g4s_plot, 0.1), alphas_T3_ref, 's--', color=CB, ms=10, lw=2,
             label='α_T³ (SIM 22, referencia)')
ax2.axhline(-1.0, color=CG, lw=1.5, ls=':', alpha=0.7, label='escala armónica α=−1')
ax2.axhline(+2.1, color=CY, lw=1.5, ls=':', alpha=0.7, label='Cap. 6 objetivo α=+2,1')
ax2.set_xlabel('g₄', color=CW); ax2.set_ylabel('α (exponente ΔE ∝ R^α)', color=CW)
ax2.set_title('Comparación directa: S³ vs T³', color=CW, fontweight='bold')
ax2.legend(facecolor=BG, labelcolor=CW, fontsize=8.5)
ax2.grid(True, alpha=0.15)

# Panel C: w_top(g₄) S³ vs T³
ax3 = axes[1, 0]
w_S3 = -alphas_S3 / 3
w_T3_ref = -alphas_T3_ref / 3
ax3.semilogx(np.maximum(g4s_plot, 0.1), w_S3, 'o-', color=CR, ms=12, lw=2.5,
             label='w_S³ = −α_S³/3')
ax3.semilogx(np.maximum(g4s_plot, 0.1), w_T3_ref, 's--', color=CB, ms=10, lw=2,
             label='w_T³ = −α_T³/3 (SIM 22)')
ax3.axhline(+1/3, color=CG, lw=1.5, ls='--', alpha=0.7, label='Radiación: +1/3')
ax3.axhline(0, color=CW, lw=1, ls='--', alpha=0.4, label='CDM: 0')
ax3.axhline(-0.70, color=CY, lw=2.5, label='Cap. 6 §6.4: w_DE = −0,70')
ax3.axhline(-1.0, color='white', lw=1.5, ls='--', alpha=0.5, label='ΛCDM: −1')
ax3.set_xlabel('g₄', color=CW); ax3.set_ylabel('w_top', color=CW)
ax3.set_title('w cosmológico: S³ vs T³\nContribución topológica al sector de energía oscura',
              color=CW, fontweight='bold')
ax3.legend(facecolor=BG, labelcolor=CW, fontsize=8)
ax3.grid(True, alpha=0.15)

# Panel D: χ_H validación
ax4 = axes[1, 1]
chi_max_S3 = np.array([ajustes[g]['chi_max'] for g in g4s_plot])
ax4.loglog(np.maximum(g4s_plot, 0.1), np.maximum(chi_max_S3, 1e-6), 'o-',
           color=CO, ms=12, lw=2.5, label='χ_H máx (S³ y R³)')
ax4.axhline(0.5, color=CR, lw=2, ls='--', label='límite Hartree')
ax4.axhline(0.1, color=CG, lw=1.5, ls=':', label='Cap. 6 plateau')
ax4.set_xlabel('g₄', color=CW); ax4.set_ylabel('χ_Hartree', color=CW)
ax4.set_title('Validez Hartree (S³)', color=CW, fontweight='bold')
ax4.legend(facecolor=BG, labelcolor=CW, fontsize=9)
ax4.grid(True, alpha=0.15, which='both')

fig.suptitle(
    f'SIM 23 — Sweep adiabático sobre S³ vs R³  vs SIM 22 (T³)\n'
    f'α_S³(g₄=0)={alpha_arm:+.3f}  α_S³(g₄_max)={alpha_max:+.3f}  '
    f'|α_S³ − α_T³| anharm máx = {delta_t3_s3_anharm:.3f}',
    fontsize=11, fontweight='bold', color=CW)
plt.tight_layout()
plt.savefig(out_path('sim23_S3_sweep.png'), dpi=150,
            bbox_inches='tight', facecolor='#0a0a1a')
plt.show()

---## 9. Resumen consolidado SIM 21 + SIM 22 + SIM 23

In [ ]:
print('═' * 80)
print('  SIM 23 — Sweep adiabático sobre S³: resumen consolidado'.center(80))
print('═' * 80)
print()
print(f'{"g₄":>8s}  {"α_S³":>10s}  {"α_T³ ref":>12s}  {"|Δα|":>8s}  '
      f'{"w_S³":>10s}  {"w_T³ ref":>12s}  {"|Δw|":>8s}')
print('─' * 80)
for g4 in sorted(by_g4.keys()):
    aS = ajustes[g4]['alpha']
    aT = ALPHA_T3_REF.get(g4, np.nan)
    wS = -aS/3
    wT = -aT/3 if not np.isnan(aT) else np.nan
    print(f'{g4:>8.0f}  {aS:>+10.4f}  {aT:>+12.4f}  {abs(aS-aT):>8.4f}  '
          f'{wS:>+10.4f}  {wT:>+12.4f}  {abs(wS-wT):>8.4f}')
print('─' * 80)

print('\nLECTURA INTEGRADA SIM 21 + SIM 22 + SIM 23:')
print()
print(f'  SIM 21 (sweep N, T³):')
print(f'    β crece de +0,98 a +1,17 con g₄ — ruptura confirmada de universalidad armónica')
print()
print(f'  SIM 22 (sweep L, T³ cosmológicamente correcto):')
print(f'    α(T³) = −1,000 a −1,291 → w_top = +0,333 a +0,430')
print(f'    Sector topológico T³ vive en w ≈ +1/3 (radiación-like)')
print()
print(f'  SIM 23 (sweep R, S³ — este test):')
print(f'    α(S³) = {alpha_arm:+.3f} a {alpha_max:+.3f} → w_top = {w_arm:+.3f} a {w_max:+.3f}')
if delta_t3_s3_anharm < 0.10:
    print(f'    Resultado CONSISTENTE con SIM 22: ambas topologías dan w ≈ +1/3')
    print(f'    → CONCLUSIÓN FINAL: el sector topológico anharmónico es UNIVERSAL,')
    print(f'      vive en w ≈ +1/3 sobre cualquier topología cerrada.')
    print(f'      La predicción w_DE = −0,70 ± 0,05 del Cap. 6 §6.4 (homogéneo) se mantiene')
    print(f'      como única predicción cosmológica del modelo y NO se ve afectada')
    print(f'      por ninguna topología cerrada bajo anharmonicidad.')
else:
    print(f'    Resultado DISTINTO de SIM 22 (|Δα|_max = {delta_t3_s3_anharm:.3f})')
    print(f'    → CONCLUSIÓN FINAL: el régimen anharmónico responde de manera específica')
    print(f'      a la topología cerrada. T³ y S³ producen w_top distintos.')
    print(f'      Esto es novedad cuantitativa del modelo que merece desarrollo en una nueva subsección.')

print()
print('Archivos generados:')
for f in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(os.path.join(OUT_DIR, f))
    print(f'  {f}  ({sz/1024:.1f} KB)')

In [ ]:
import subprocess
subprocess.run(['zip', '-q', '-r', os.path.join(OUT_DIR, 'sim23_resultados.zip'), '.'], cwd=OUT_DIR)
print(f'\n→ {OUT_DIR}/sim23_resultados.zip')

---## Qué hacer con los resultados1. Descargar `sim23_resultados.zip`2. Pasarme el `.png` principal y la salida de texto del resumen consolidado SIM 21 + 22 + 233. Con eso cerramos definitivamente el cuadro del sector topológico anharmónico:   - **Si |Δα| < 0,10 (universalidad confirmada):** subsección 6.3bis "Test extendido: topología anharmónica" con T³ y S³, mostrando que el sector topológico vive en w ≈ +1/3 universalmente. Refuerza Cap. 6 §6.4.   - **Si |Δα| > 0,10 (dependencia topológica):** subsección nueva con mayor desarrollo, posiblemente con implicación para selección topológica del universo (Ω_k ≈ 0 reforzado por preferencia anharmónica de T³ sobre S³ en términos cosmológicos).**Si Hartree no converge para g₄ alto en S³** (el sector IR de S³ puede ser más sensible), bajar `g₄_max` a 2500 o subir `HARTREE_MAX` a 40.